In [1]:
# Cài đặt các thư viện bắt buộc cho Pipeline và ép cập nhật torchao
!pip install -q -U "torchao>0.16.0"
!pip install -q open_clip_torch
!pip install -q evaluate
!pip install -q rouge_score
!pip install -q bert_score
!pip install -q scikit-learn
!pip install evaluate rouge_score bert_score scikit-learn
!pip install -q bitsandbytes accelerate
!pip install -q -U transformers accelerate datasets
!pip install -q einops
print("✅ Đã cài đặt xong tất cả các thư viện cần thiết và cập nhật torchao!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
✅ Đã cài đặt xong tất cả các thư viện cần thiết và cập nhật torchao!


In [2]:
import os
import gc
import time
import torch
import re
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from datasets import load_dataset
import transformers
from transformers import AutoModel, AutoTokenizer
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
import bitsandbytes as bnb 
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
import evaluate
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm 
import warnings
import ctypes
import urllib.request
import zipfile
from peft import LoraConfig, get_peft_model

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ==========================================
# 0. VÁ LỖI KIẾN TRÚC INTERNVL & HUGGINGFACE
# ==========================================
transformers.PreTrainedModel.all_tied_weights_keys = {}
transformers.logging.set_verbosity_error()
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.8"
torch.backends.cudnn.benchmark = True 

runName = "FedMVQA_VQARAD_InternVL3.5_LoRA_448_Alpha1_R1_E2"
kaggleBaseDir = "/kaggle/working/Fed-MVQA-Project"
dataDir = os.path.join(kaggleBaseDir, "data", "vqarad")
logDir = os.path.join(kaggleBaseDir, "logs")
checkpointDir = os.path.join(kaggleBaseDir, "checkpoints", runName)

os.makedirs(dataDir, exist_ok=True)
os.makedirs(logDir, exist_ok=True)
os.makedirs(checkpointDir, exist_ok=True)

logPath = os.path.join(logDir, f"{runName}_log.csv")
checkpointPath = os.path.join(checkpointDir, "best_fed_mvqa_model_lora.pt")

# ==========================================
# 1. CÔNG CỤ QUẢN LÝ BỘ NHỚ (OOM PREVENTION)
# ==========================================
def force_garbage_collect():
    """Hút cạn sạch bộ nhớ rác của Python và C++ để chống sập RAM Kaggle"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    try:
        libc = ctypes.CDLL("libc.so.6")
        libc.malloc_trim(0)
    except Exception:
        pass

def getPeakReservedMemory():
    return f"{torch.cuda.memory_reserved() / (1024 ** 3):.2f} GB" if torch.cuda.is_available() else "0.00 GB"

def calculateCommunicationMetrics(trainableParams, numClients):
    peftBytes = trainableParams * 2 # FP16 = 2 bytes
    fullRoundMb = (peftBytes * 2 * numClients) / (1024 * 1024)
    return fullRoundMb

def processImage(pilImg, targetSize=448):
    if pilImg.mode != "RGB": 
        pilImg = pilImg.convert("RGB")
    imageTransform = T.Compose([
        T.Resize((targetSize, targetSize), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    ])
    return imageTransform(pilImg).unsqueeze(0)

# ==========================================
# 2. KHỞI TẠO MÔ HÌNH TOÀN CỤC & TIÊM LORA
# ==========================================
print("--- KHỞI TẠO HẠ TẦNG FED-LORA (INTERNVL3.5-2B, 448x448, FP16) ---")
force_garbage_collect()

modelId = "OpenGVLab/InternVL3_5-2B-Instruct"
baseModel = AutoModel.from_pretrained(
    modelId,
    torch_dtype=torch.float16, 
    trust_remote_code=True,
    low_cpu_mem_usage=True 
).cuda()

tokenizer = AutoTokenizer.from_pretrained(modelId, trust_remote_code=True)
print("✅ Đã nạp thành công mô hình Base InternVL3.5-2B!")

# VÁ LỖI CỐT LÕI CHỐNG OOM VRAM: Bật Gradient Checkpointing
baseModel.config.use_cache = False
if hasattr(baseModel, 'enable_input_require_grads'):
    baseModel.enable_input_require_grads()

try:
    baseModel.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
except: pass

if hasattr(baseModel, 'language_model'):
    baseModel.language_model.gradient_checkpointing_enable()
if hasattr(baseModel, 'vision_model'):
    baseModel.vision_model.gradient_checkpointing_enable()

# ĐÓNG BĂNG BASE MODEL
for p in baseModel.parameters():
    p.requires_grad = False

# CẤU HÌNH VÀ TIÊM LORA
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none"
)

baseModel = get_peft_model(baseModel, lora_config)
print("✅ Đã tiêm cấu trúc LoRA thành công!")

trainableParams = sum(p.numel() for p in baseModel.parameters() if p.requires_grad)
totalModelParams = sum(p.numel() for p in baseModel.parameters())
print(f"[Metrics] Trainable Params (LoRA): {trainableParams:,} ({100 * trainableParams / totalModelParams:.2f}%)")

# TRÍCH XUẤT LORA WEIGHTS LÀM GLOBAL WEIGHTS
globalWeights = {k: v.detach().cpu().clone() for k, v in baseModel.state_dict().items() if "lora" in k}
force_garbage_collect()

# ==========================================
# 3. TRÍCH XUẤT ĐỘNG ID TOKEN HÌNH ẢNH 
# ==========================================
print("Đang trích xuất tự động Image Context ID từ lõi mô hình...")
interceptedIds = None
originalForward = baseModel.forward

def hookForward(*args, **kwargs):
    global interceptedIds
    if 'input_ids' in kwargs and kwargs['input_ids'] is not None:
        interceptedIds = kwargs['input_ids'].detach().cpu()
    elif len(args) > 1 and args[1] is not None:
        interceptedIds = args[1].detach().cpu()
    raise RuntimeError("STOP_INTERCEPT")

baseModel.forward = hookForward
try:
    dummyPx = torch.zeros((1, 3, 448, 448), dtype=torch.float16).cuda()
    baseModel.chat(tokenizer, dummyPx, "", generation_config=dict(max_new_tokens=1))
except RuntimeError as e:
    if str(e) != "STOP_INTERCEPT": pass
except Exception: pass
finally:
    baseModel.forward = originalForward

if interceptedIds is not None:
    counts = torch.bincount(interceptedIds.view(-1))
    imgContextId = int(counts.argmax().item())
else:
    imgContextId = getattr(baseModel, 'img_context_token_id', tokenizer.convert_tokens_to_ids(''))
    
print(f"✅ Đã tìm thấy Image Context ID an toàn: {imgContextId}")

# ==========================================
# 4. CHUẨN BỊ DỮ LIỆU VQA-RAD (ALPHA = 1.0, 5 CLIENTS)
# ==========================================
print("\n--- CHUẨN BỊ DỮ LIỆU VQA-RAD (NEAR-IID α=1.0) ---")

dataset = load_dataset("flaviagiammarino/vqa-rad", cache_dir=dataDir)
shuffledTrain = dataset["train"].shuffle(seed=42)
testDataset = dataset["test"]

def partitionDataNonIidDirichlet(datasetInput, numClients=5, alpha=1.0):
    labels = [str(item["answer"]).strip().lower() for item in datasetInput]
    uniqueLabels = list(set(labels))
    clientIndices = {i: [] for i in range(numClients)}
    for label in uniqueLabels:
        labelIdxs = [i for i, l in enumerate(labels) if l == label]
        np.random.shuffle(labelIdxs)
        proportions = np.random.dirichlet([alpha] * numClients)
        splits = np.split(labelIdxs, (np.cumsum(proportions)[:-1] * len(labelIdxs)).astype(int))
        for i in range(numClients): 
            clientIndices[i].extend(splits[i].tolist())
            
    return [datasetInput.select(np.random.permutation(clientIndices[i])) for i in range(numClients)]

clientNames = ["A", "B", "C", "D", "E"]
clientDatasets = partitionDataNonIidDirichlet(shuffledTrain, numClients=5, alpha=1.0)
trainLoaders = []
dataSizes = [len(ds) for ds in clientDatasets]

print(f"Dữ liệu Client A: {dataSizes[0]} mẫu")
print(f"Dữ liệu Client B: {dataSizes[1]} mẫu")
print(f"Dữ liệu Client C: {dataSizes[2]} mẫu")
print(f"Dữ liệu Client D: {dataSizes[3]} mẫu")
print(f"Dữ liệu Client E: {dataSizes[4]} mẫu")
print(f"Dữ liệu Global Test: {len(testDataset)} mẫu")

systemInstruction = (
    "You are an expert medical diagnostic assistant. Your task is to answer clinical questions "
    "based EXCLUSIVELY on the provided medical image.\nStrictly follow these rules:\n"
    "1. Analyze the provided image.\n"
    "2. Provide a concise and definitive answer (1-3 words if possible)."
)

def customCollateFn(batch):
    pixelValuesList, inputIdsList, labelsList = [], [], []
    for item in batch:
        img = item["image"].convert("RGB") 
        q, a = item["question"], str(item["answer"]).strip()
        
        promptPart1 = f"{systemInstruction}\nUser: "
        promptPart2 = f"\nQuestion: {q}\nFinal Answer:\nAssistant: "

        part1Ids = tokenizer(promptPart1, return_tensors="pt", add_special_tokens=True).input_ids[0]
        part2Ids = tokenizer(promptPart2, return_tensors="pt", add_special_tokens=False).input_ids[0]
        answerIds = tokenizer(a + getattr(tokenizer, 'eos_token', ''), return_tensors="pt", add_special_tokens=False).input_ids[0]
        
        imgTokens = torch.full((256,), imgContextId, dtype=torch.long)
        inputIds = torch.cat([part1Ids, imgTokens, part2Ids, answerIds], dim=0)
        
        lbls = inputIds.clone()
        promptLen = len(part1Ids) + 256 + len(part2Ids)
        lbls[:promptLen] = -100 
        
        inputIdsList.append(inputIds)
        labelsList.append(lbls)
        pixelValuesList.append(processImage(img))

    maxLen = max(len(ids) for ids in inputIdsList)
    padId = getattr(tokenizer, 'pad_token_id', getattr(tokenizer, 'eos_token_id', 0))
    if padId is None: 
        padId = 0

    paddedInputIds, paddedLabels, attentionMask = [], [], []
    for ids, lbls in zip(inputIdsList, labelsList):
        padLen = maxLen - len(ids)
        paddedInputIds.append(torch.cat([ids, torch.full((padLen,), padId, dtype=torch.long)]))
        paddedLabels.append(torch.cat([lbls, torch.full((padLen,), -100, dtype=torch.long)]))
        attentionMask.append(torch.cat([torch.ones(len(ids), dtype=torch.long), torch.zeros(padLen, dtype=torch.long)]))

    return {
        "input_ids": torch.stack(paddedInputIds),
        "labels": torch.stack(paddedLabels),
        "attention_mask": torch.stack(attentionMask),
        "pixel_values": torch.cat(pixelValuesList, dim=0).to(torch.float16),
        "image_flags": torch.ones((len(batch), 1), dtype=torch.long) 
    }

batchSize = 1
for ds in clientDatasets:
    trainLoaders.append(DataLoader(
        ds,
        batch_size=batchSize,
        shuffle=True,
        collate_fn=customCollateFn,
        num_workers=0,
        pin_memory=False
    ))

# ==========================================
# 5. HÀM HUẤN LUYỆN CỤC BỘ (ZERO-PAGE RAM OPTIMIZED LORA)
# ==========================================
def trainLocalClient(model, dataLoader, clientId, roundIdx, epochs=2, learningRate=2e-4):
    model.train()

    decayFactor = 0.9 ** (roundIdx - 1)
    activeLr = learningRate * decayFactor

    optimizer = bnb.optim.PagedAdamW8bit(model.parameters(), lr=activeLr, weight_decay=0.01)
    accumulationSteps = 4 
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs * len(dataLoader) // accumulationSteps)

    print(f" [Client {clientId}] Bắt đầu huấn luyện LoRA (PagedAdamW8bit, LR={activeLr:.2e})...")
    startTime = time.time()
    totalLoss = 0

    for epoch in range(epochs):
        optimizer.zero_grad(set_to_none=True)
        progressLoop = tqdm(dataLoader, desc=f"Client {clientId} | Epoch {epoch+1}/{epochs}", leave=False, dynamic_ncols=True)
        for step, batch in enumerate(progressLoop):
            batch = {k: v.cuda(non_blocking=True) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            
            with torch.amp.autocast("cuda", dtype=torch.float16):
                outputs = model(**batch)
                loss = outputs.loss / accumulationSteps
            
            loss.backward()
            
            del outputs, loss
            
            if (step + 1) % accumulationSteps == 0 or (step + 1) == len(dataLoader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                
            totalLoss += (loss.item() * accumulationSteps) if 'loss' in locals() else 0
            
            del batch
            if step % 20 == 0:  
                force_garbage_collect()
            
    trainDuration = time.time() - startTime
    avgLoss = totalLoss / (len(dataLoader) * epochs)
    peakVram = getPeakReservedMemory()

    del optimizer, scheduler
    force_garbage_collect()

    return trainDuration, avgLoss, peakVram

# ==========================================
# 6. HÀM ĐÁNH GIÁ
# ==========================================
def cleanOpenAnswer(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    stopwords = {'a', 'an', 'the', 'is', 'are', 'was', 'were', 'it', 'there', 'shows', 'showing', 'of', 'in', 'on', 'to'}
    words = text.split()
    words = [w for w in words if w not in stopwords]
    return " ".join(words)

def evaluateGlobalModel(model, datasetInput):
    model.eval()
    print(f"\n [Server] Đang đánh giá trên tập Test ({len(datasetInput)} mẫu)...")

    openPreds, openRefs, closePreds, closeRefs, inferenceTimes = [], [], [], [], []
    evalLoop = tqdm(datasetInput, desc="Evaluating", leave=False, dynamic_ncols=True)

    for item in evalLoop:
        q, gtAnswer = item["question"], str(item["answer"]).strip().lower()
        ansType = str(item.get("answer_type", "CLOSED" if gtAnswer in ["yes", "no"] else "OPEN")).strip().upper()
        
        promptText = f"{systemInstruction}\n[Target Case Image]\n\n\nCurrent Question: {q}\nBased on the visual evidence, what is the correct answer?\nFinal Answer: "
        img = item["image"].convert("RGB")
        pixelValues = processImage(img).to(torch.float16).cuda()
        
        startTime = time.time()
        with torch.inference_mode():
            aiAnswerRaw = model.chat(
                tokenizer=tokenizer,
                pixel_values=pixelValues,
                question=promptText,
                generation_config=dict(max_new_tokens=15, do_sample=False)
            )
        inferenceTimes.append(time.time() - startTime)
        aiAnswer = aiAnswerRaw.lower().strip()
        
        if ansType == "CLOSED":
            predClean = 'yes' if 'yes' in aiAnswer and 'no' not in aiAnswer else ('no' if 'no' in aiAnswer and 'yes' not in aiAnswer else ('yes' if aiAnswer.find('yes') < aiAnswer.find('no') else 'no'))
            closePreds.append(predClean)
            closeRefs.append('yes' if 'yes' in gtAnswer else ('no' if 'no' in gtAnswer else gtAnswer))
        else:
            cleanStr = lambda t: " ".join([w for w in re.sub(r'[^\w\s]', '', t.lower()).split() if w not in {'a', 'an', 'the', 'is', 'are', 'was', 'were', 'it', 'there', 'shows', 'showing', 'of', 'in', 'on', 'to'}])
            openPreds.append(cleanStr(aiAnswer) if cleanStr(aiAnswer) else "none")
            openRefs.append(cleanStr(gtAnswer) if cleanStr(gtAnswer) else "none")
            
        del pixelValues
        torch.cuda.empty_cache()
        
    res = {"avgInferenceTime": np.mean(inferenceTimes), "overallAccuracy": (sum(p==r for p, r in zip(closePreds, closeRefs)) + sum(p==r for p, r in zip(openPreds, openRefs))) / len(datasetInput)}
    if closeRefs: 
        res["Closed"] = {"ACC": accuracy_score(closeRefs, closePreds), "F1": f1_score(closeRefs, closePreds, average='weighted', zero_division=0), "count": len(closeRefs)}
    if openRefs:
        bertRes = evaluate.load("bertscore").compute(predictions=openPreds, references=openRefs, lang="en")
        res["Open"] = {"BLEU-1": evaluate.load("bleu").compute(predictions=openPreds, references=[[r] for r in openRefs], max_order=1)["bleu"], "ROUGE-L": evaluate.load("rouge").compute(predictions=openPreds, references=[[r] for r in openRefs])["rougeL"], "BERTScore": np.mean(bertRes["f1"]), "count": len(openRefs)}
    return res

# ==========================================
# 7. VÒNG LẶP HỌC LIÊN KẾT CHÍNH (FED-LORA OPTIMIZED)
# ==========================================
maxRounds = 1
localEpochs = 2
performanceLog = []
cumulativeCommGb = 0.0
totalSamples = sum(dataSizes)

print(f"\n=== BẮT ĐẦU HUẤN LUYỆN FED-LORA INTERNVL3.5-2B (VQA-RAD, {maxRounds} ROUNDS, {localEpochs} EPOCHS/CLIENT, ALPHA=1.0) ===")

for roundIdx in range(1, maxRounds + 1):
    print(f"\n" + "="*50)
    print(f"--- COMMUNICATION ROUND {roundIdx} ---")
    print("="*50)

    latestTrainTimes, roundLoss = [], 0
    peakVramRecorded = "0.00 GB"

    print("\n [Server] Đang thiết lập cấu trúc In-Place Accumulation cho LoRA...")
    backupWeights = {k: v.clone() for k, v in globalWeights.items()}

    for k in globalWeights.keys():
        if globalWeights[k].dtype.is_floating_point:
            globalWeights[k].zero_()
            
    for i, clientId in enumerate(clientNames):
        baseModel.load_state_dict(backupWeights, strict=False)
        force_garbage_collect()
        
        t, l, vram = trainLocalClient(baseModel, trainLoaders[i], clientId, roundIdx=roundIdx, epochs=localEpochs, learningRate=2e-4)
        latestTrainTimes.append(t)
        roundLoss += l
        if clientId == "E": 
            peakVramRecorded = vram
        
        weightRatio = dataSizes[i] / totalSamples
        client_state_dict = {k: v for k, v in baseModel.state_dict().items() if "lora" in k}
        
        for k, v in client_state_dict.items():
            if v.dtype.is_floating_point:
                globalWeights[k].add_(v.detach().cpu().float() * weightRatio)
        
        force_garbage_collect()

    print("\n [Server] Hoàn tất Aggregation cho LoRA. Ép kiểu về Float16...")
    for k, v in globalWeights.items():
        if v.dtype.is_floating_point:
            globalWeights[k] = v.to(torch.float16)
            
    del backupWeights
    force_garbage_collect()

    baseModel.load_state_dict(globalWeights, strict=False)
    force_garbage_collect()

    # Bật cache để tăng tốc quá trình đánh giá
    baseModel.config.use_cache = True
    evalRes = evaluateGlobalModel(baseModel, testDataset)
    baseModel.config.use_cache = False

    roundAccuracy = evalRes["overallAccuracy"] * 100
    roundAccClosed = evalRes.get("Closed", {}).get("ACC", 0) * 100
    roundF1Closed = evalRes.get("Closed", {}).get("F1", 0) * 100
    roundBleu = evalRes.get("Open", {}).get("BLEU-1", 0)
    roundRouge = evalRes.get("Open", {}).get("ROUGE-L", 0)
    roundBert = evalRes.get("Open", {}).get("BERTScore", 0)

    fullRoundMb = calculateCommunicationMetrics(trainableParams, 5)
    cumulativeCommGb += (fullRoundMb / 1024)
    avgTrainingTime = np.mean(latestTrainTimes)
    averageRoundLoss = roundLoss / 5

    roundLog = {
        "Round": roundIdx,
        "Avg Loss": round(averageRoundLoss, 4),
        "Overall Acc (%)": round(roundAccuracy, 2),
        "Closed Acc (%)": round(roundAccClosed, 2),
        "Closed F1 (%)": round(roundF1Closed, 2),
        "Open BLEU-1": round(roundBleu, 4),
        "Open ROUGE-L": round(roundRouge, 4),
        "Open BERTScore": round(roundBert, 4),
        "MB per Round": round(fullRoundMb, 2),
        "Cumul Data (GB)": round(cumulativeCommGb, 4),
        "Avg Train Time (s)": round(avgTrainingTime, 2)
    }
    performanceLog.append(roundLog)

    print(f"\n[VÒNG {roundIdx}] Loss: {averageRoundLoss:.4f} | Acc: {roundAccuracy:.2f}% | Closed Acc: {roundAccClosed:.2f}%, F1: {roundF1Closed:.2f}% | Open BLEU-1: {roundBleu:.4f}, ROUGE-L: {roundRouge:.4f} | VRAM: {peakVramRecorded}")

    pd.DataFrame(performanceLog).to_csv(logPath, index=False)

    if roundIdx == maxRounds:
        torch.save(globalWeights, checkpointPath)
        print(f"\n   -> [Cập nhật] Mô hình tốt nhất đã được lưu tại Round {roundIdx}.")

# ==========================================
# 8. BÁO CÁO TỔNG KẾT CHI TIẾT
# ==========================================
print("\n" + "="*80)
print(f" FINAL SUMMARY REPORT: FED-LORA INTERNVL3.5 (VQA-RAD, {maxRounds} ROUNDS, {localEpochs} EPOCHS, ALPHA=1.0) ")
print("="*80)

print("\n[1] PERFORMANCE & HARDWARE METRICS")
print(f"  + Trainable Parameters   : {trainableParams:,} ({100 * trainableParams / totalModelParams:.2f}%)")
print(f"  + Peak Reserved Memory   : {peakVramRecorded}")
print(f"  + Average Inference Time : {evalRes['avgInferenceTime']:.4f} giây/ảnh")

print("\n[2] CLINICAL EVALUATION METRICS (GLOBAL MODEL VÒNG CUỐI)")
if "Closed" in evalRes:
    print(f"  - CLOSED QUESTIONS ({evalRes['Closed'].get('count', 'N/A')} samples):")
    print(f"    * Accuracy : {evalRes['Closed']['ACC']*100:.2f}%")
    print(f"    * F1-Score : {evalRes['Closed']['F1']*100:.2f}%")
if "Open" in evalRes:
    print(f"  - OPEN QUESTIONS ({evalRes['Open'].get('count', 'N/A')} samples):")
    print(f"    * BLEU-1   : {evalRes['Open']['BLEU-1']:.4f}")
    print(f"    * ROUGE-L  : {evalRes['Open']['ROUGE-L']:.4f}")
    print(f"    * BERTScore: {evalRes['Open']['BERTScore']:.4f}")

print("\n[3] CONVERGENCE LOG CHI TIẾT")
commTable = pd.DataFrame(performanceLog)
print(commTable[['Round', 'Avg Loss', 'Overall Acc (%)', 'Closed Acc (%)', 'Open BLEU-1', 'MB per Round', 'Cumul Data (GB)', 'Avg Train Time (s)']].to_string(index=False))

print("\n" + "="*80)
print(" QUÁ TRÌNH THỰC NGHIỆM ĐÃ HOÀN TẤT THÀNH CÔNG ")
print("="*80)

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


--- KHỞI TẠO HẠ TẦNG FED-LORA (INTERNVL3.5-2B, 448x448, FP16) ---


config.json:   0%|          | 0.00/5.75k [00:00<?, ?B/s]

configuration_internvl_chat.py:   0%|          | 0.00/4.70k [00:00<?, ?B/s]

configuration_intern_vit.py:   0%|          | 0.00/5.55k [00:00<?, ?B/s]

modeling_internvl_chat.py:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

... (42 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]


[VÒNG 1] Loss: 0.0000 | Acc: 51.44% | Closed Acc: 70.92%, F1: 69.48% | Open BLEU-1: 0.2482, ROUGE-L: 0.3522 | VRAM: 5.95 GB

   -> [Cập nhật] Mô hình tốt nhất đã được lưu tại Round 1.

 FINAL SUMMARY REPORT: FED-LORA INTERNVL3.5 (VQA-RAD, 1 ROUNDS, 2 EPOCHS, ALPHA=1.0) 

[1] PERFORMANCE & HARDWARE METRICS
  + Trainable Parameters   : 34,865,152 (1.46%)
  + Peak Reserved Memory   : 5.95 GB
  + Average Inference Time : 0.3391 giây/ảnh

[2] CLINICAL EVALUATION METRICS (GLOBAL MODEL VÒNG CUỐI)
  - CLOSED QUESTIONS (251 samples):
    * Accuracy : 70.92%
    * F1-Score : 69.48%
  - OPEN QUESTIONS (200 samples):
    * BLEU-1   : 0.2482
    * ROUGE-L  : 0.3522
    * BERTScore: 0.9042

[3] CONVERGENCE LOG CHI TIẾT
 Round  Avg Loss  Overall Acc (%)  Closed Acc (%)  Open BLEU-1  MB per Round  Cumul Data (GB)  Avg Train Time (s)
     1       0.0            51.44           70.92       0.2482         665.0           0.6494              613.75

 QUÁ TRÌNH THỰC NGHIỆM ĐÃ HOÀN TẤT THÀNH CÔNG 
